In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# エラー軽減・抑制技術

> **Note:** 新しい実行モデルのベータ版が利用可能になりました。指向型実行モデルは、エラー軽減ワークフローのカスタマイズにより高い柔軟性を提供します。詳細は[指向型実行モデル](/guides/directed-execution-model)ガイドを参照してください。


<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは以下の要件を使用して開発されました。
これらのバージョン以降を使用することをお勧めします。

```
qiskit-ibm-runtime~=0.43.1
```
</details>
エラー軽減とエラー抑制技術は、より大規模なワークロードへのスケールアップ時に結果品質を向上させるために使用されます。このページでは、Qiskit Runtimeで利用可能なエラー抑制・エラー軽減技術について高レベルな解説を提供します。

以下のセルは、Estimatorプリミティブをインポートし、後のコードセルでEstimatorの初期化に使用するバックエンドを作成します。

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## ダイナミカル・デカップリング
量子Circuitは、IBM&reg;ハードウェア上でマイクロ波パルスのシーケンスとして実行され、精密な時間間隔でスケジュールおよび実行される必要があります。
残念ながら、Qubit間の望ましくない相互作用によって、アイドル状態のQubitにコヒーレントエラーが生じる可能性があります。ダイナミカル・デカップリングは、アイドル状態のQubitにパルスシーケンスを挿入することでこれらのエラーの影響をほぼ相殺する手法です。挿入された各パルスシーケンスは恒等演算に相当しますが、パルスの物理的な存在がエラーを抑制する効果をもたらします。
パルスシーケンスの選択肢は多数あり、特定のケースにどのシーケンスが優れているかは[活発な研究分野](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027)として続いています。

ダイナミカル・デカップリングは主に、一部のQubitが操作なしにアイドル状態になるギャップを含むCircuitに対して有効です。Circuit内の操作が非常に密に詰まっており、ほぼ常にすべてのQubitが使用中の場合、ダイナミカル・デカップリングパルスの追加はパフォーマンスを向上させないかもしれません。実際、パルス自体の不完全さにより、パフォーマンスが悪化する可能性もあります。

下図は、XXパルスシーケンスによるダイナミカル・デカップリングを示しています。左側の抽象Circuitは、右上のマイクロ波パルススケジュールにマッピングされています。右下は同じスケジュールを示していますが、最初のQubitのアイドル期間に2つのXパルスのシーケンスが挿入されています。

![ダイナミカル・デカップリングの描写](../docs/images/guides/error-mitigation-explanation/dd.avif)

ダイナミカル・デカップリングは、[ダイナミカル・デカップリングオプション](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options)の`enable`を`True`に設定することで有効にできます。`sequence_type`オプションを使用して、いくつかの異なるパルスシーケンスから選択できます。デフォルトのシーケンスタイプは`"XX"`です。

以下のコードセルは、EstimatorのダイナミカルデカップリングをどのようにしてXを有効にし、ダイナミカル・デカップリングシーケンスを選択するかを示しています。

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## パウリ・トワーリング
トワーリングは[ランダム化コンパイル](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325)とも呼ばれ、任意のノイズチャネルをより特定の構造を持つノイズチャネルに変換するために広く使用されている技術です。

パウリ・トワーリングは、パウリ演算を使用するトワーリングの特別な形式です。これにより、任意の量子チャネルをパウリチャネルに変換する効果があります。単独で行われる場合、コヒーレントノイズを軽減できます。コヒーレントノイズは演算数の二乗で累積する傾向があるのに対し、パウリノイズは線形に累積するためです。パウリ・トワーリングは、任意のノイズよりもパウリノイズでより効果的に機能する他のエラー軽減技術と組み合わせて使用されることが多いです。

パウリ・トワーリングは、選択したゲートのセットをランダムに選ばれた1量子ビットパウリGateで挟むことで実装され、ゲートの理想的な効果が同じになるように設計されています。その結果、単一のCircuitは、すべて同じ理想的な効果を持つCircuitのランダムなアンサンブルに置き換えられます。Circuitをサンプリングする際、単一のインスタンスだけでなく、複数のランダムなインスタンスからサンプルが取得されます。

![パウリ・トワーリングの描写](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

現在の量子ハードウェアにおけるエラーのほとんどが2量子ビットGateから来るため、この技術は（ネイティブの）2量子ビットGateにのみ適用されることが多いです。下図は、CNOTゲートとECRゲートのいくつかのパウリ・トワールを示しています。各行内のすべてのCircuitは同じ理想的な効果を持ちます。

![ゲート・トワールの描写](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

パウリ・トワーリングは、[トワーリングオプション](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)の`enable_gates`を`True`に設定することで有効にできます。その他の注目すべきオプションには以下が含まれます：

- `num_randomizations`：トワーリングされたCircuitのアンサンブルから取得するCircuitインスタンスの数。
- `shots_per_randomization`：各Circuitインスタンスからサンプリングするショット数。

以下のコードセルは、Estimatorのパウリ・トワーリングを有効にし、これらのオプションを設定する方法を示しています。これらのオプションは明示的に設定する必要はありません。

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## トワーリング読み出しエラー消去（TREX）
[トワーリング読み出しエラー消去（TREX）](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620)は、パウリオブザーバブルの期待値推定における測定エラーの影響を軽減します。
これはトワーリング測定の概念に基づいており、測定Gateをランダムに（1）パウリXゲート、（2）測定、（3）古典ビットフリップのシーケンスで置き換えることで実現します。標準的なゲートトワーリングと同様に、このシーケンスはノイズがない場合には単純な測定と等価であり、以下の図に示されています：

![測定トワーリングの描写](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

読み出しエラーが存在する場合、測定トワーリングは読み出しエラー転送行列を対角化する効果があり、容易に逆行列を計算できます。読み出しエラー転送行列の推定には追加のキャリブレーションCircuitの実行が必要であり、わずかなオーバーヘッドが生じます。

TREXは、Estimatorの[Qiskit Runtimeレジリエンスオプション](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2)で`measure_mitigation`を`True`に設定することで有効にできます。測定ノイズ学習のオプションは[こちら](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)に説明されています。ゲートトワーリングと同様に、Circuitのランダム化数とランダム化ごとのショット数を設定できます。

以下のコードセルは、TREXを有効にし、Estimatorのこれらのオプションを設定する方法を示しています。これらのオプションは明示的に設定する必要はありません。

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## ゼロノイズ外挿（ZNE）
ゼロノイズ外挿（ZNE）は、オブザーバブルの期待値推定におけるエラーを軽減するための技術です。しばしば結果を改善しますが、偏りのない結果を保証するものではありません。

ZNEは2つの段階で構成されます：

1. _ノイズ増幅_：元の量子Circuitが異なるノイズレートで複数回実行されます。
2. _外挿_：ノイズのある期待値の結果をゼロノイズ極限に外挿することで理想的な結果を推定します。

ノイズ増幅と外挿の両段階は、多くの異なる方法で実装できます。Qiskit Runtimeは「デジタルゲートフォールディング」によってノイズ増幅を実装します。これは、2量子ビットGateをゲートとその逆行列の等価なシーケンスで置き換えることを意味します。例えば、ユニタリ $U$ を $U U^\dagger U$ に置き換えると、ノイズ増幅係数3が得られます。外挿には、線形フィットや指数フィットなど、いくつかの関数形式から選択できます。
下図は、左側にデジタルゲートフォールディングを、右側に外挿手順を示しています。

![ZNEの描写](../docs/images/guides/error-mitigation-explanation/zne.avif)

ZNEは、Estimatorの[Qiskit Runtimeレジリエンスオプション](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2)で`zne_mitigation`を`True`に設定することで有効にできます。
ZNEのQiskit Runtimeオプションは[こちら](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)に説明されています。以下のオプションが注目されます：

- `noise_factors`：ノイズ増幅に使用するノイズ係数。
- `extrapolator`：外挿に使用する関数形式。

以下のコードセルは、ZNEを有効にし、Estimatorのこれらのオプションを設定する方法を示しています。これらのオプションは明示的に設定する必要はありません。

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## 確率的エラー増幅（PEA）
ZNEにおける主要な課題の1つは、対象Circuitに影響するノイズを正確に増幅することです。ゲートフォールディングはこの増幅を行う簡単な方法を提供しますが、不正確になる可能性があり、誤った結果につながる場合があります。詳細については["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197)の論文、特に補足情報の4ページを参照してください。確率的エラー増幅は、ノイズ学習を通じてより正確なエラー増幅アプローチを提供します。

PEAは、まず予備実験を行ってノイズを再構築し、この情報を使用して正確な増幅を実行するより洗練された技術です。Circuit内の各もつれゲート層のトワーリングされたノイズモデルを（実行前に）学習することから始まります（関連する学習オプションについては[LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)を参照）。学習フェーズの後、各ノイズ係数でCircuitが実行されます。Circuitの各もつれゲート層は、対応する学習済みノイズモデルに比例した1量子ビットノイズを確率的に注入することで増幅されます。詳細については["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3)の論文を参照してください。

PEAは3つの段階で構成されます：
1. _学習_：Circuit内の各もつれゲート層のトワーリングされたノイズモデルが学習されます。
1. _ノイズ増幅_：元の量子Circuitが異なるノイズ係数で複数回実行されます。
2. _外挿_：ノイズのある期待値の結果をゼロノイズ極限に外挿することで理想的な結果を推定します。

ユーティリティスケールの実験では、PEAが最適な選択肢であることが多いです。

PEAはZNEノイズ増幅技術であるため、`resilience.zne_mitigation = True`を設定してZNEも有効にする必要があります。他の[`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)オプションを追加で使用して、外挿器、増幅レベルなどを設定することができます。PEAはノイズモデルを必要とし、プリミティブを使用する際に自動的に生成されます。

以下のスニペットは、PEAを使用してEstimatorジョブの結果を軽減する例を示しています：

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## 確率的エラーキャンセル（PEC）
確率的エラーキャンセル（PEC）は、オブザーバブルの期待値推定におけるエラーを軽減するための技術です。ZNEとは異なり、期待値の偏りのない推定を返します。ただし、一般的により大きなオーバーヘッドが発生します。

PECでは、理想的な対象Circuitの効果が、実際に実装可能なノイズのあるCircuitの線形結合として表されます：

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

理想的なCircuitの出力は、線形結合によって定義されたランダムアンサンブルから取得された異なるノイズありCircuitインスタンスを実行することで再現できます。係数 $\eta_i$ が確率分布を形成する場合、アンサンブルの確率として直接使用できます。実際には、一部の係数が負であるため、代わりに準確率分布を形成します。これらは依然としてランダムアンサンブルを定義するために使用できますが、準確率分布の負性に関連するサンプリングオーバーヘッドが存在し、次の量で特徴付けられます：

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

サンプリングオーバーヘッドは、理想のCircuitから必要なショット数と比較して、特定の精度で期待値を推定するために必要なショット数に対する乗数係数です。これは $\gamma$ の二乗でスケールし、 $\gamma$ 自体はCircuitの深さに対して指数的にスケールします。

PECは、Estimatorの[Qiskit Runtimeレジリエンスオプション](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2)で`pec_mitigation`を`True`に設定することで有効にできます。
PECのQiskit Runtimeオプションは[こちら](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)に説明されています。サンプリングオーバーヘッドの上限は`max_overhead`オプションを使用して設定できます。サンプリングオーバーヘッドを制限すると、結果の精度が要求された精度を超える場合があります。`max_overhead`のデフォルト値は100です。

以下のコードセルは、PECを有効にし、Estimatorの`max_overhead`オプションを設定する方法を示しています。

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## 次のステップ
- Estimatorプリミティブでエラー軽減オプションを組み合わせる[チュートリアル](/tutorials/combine-error-mitigation-techniques)をご確認ください。
- [エラー軽減の設定。](configure-error-mitigation)
- [エラー抑制の設定。](configure-error-suppression)
- Qiskit Runtimeプリミティブの他の[オプション](runtime-options-overview)を探索してください。
- ジョブを実行する[実行モード](execution-modes)を決定してください。